# Instrument Configuration
This notebook demonstrates how to explore, create, and customize instrument configuration files.

## Understanding Paths
Before creating configurations, it is important to understand where files are stored:
- **Host Path:** The folder on your physical machine defined by `INSTRUMENT_CONFIG_PATH` in your `.env` file.
- **Container Path:** Inside this Jupyter environment, that folder is mounted at `/data/instrument_configs`.

Any JSON file you create or edit here will be immediately available in your host folder.

## Displaying the Default Configuration
The following code displays the default configuration included in the library by directly reading the `default.json` file, including comments.

Note that the default configuration includes sections for `header_keywords`, `camera_specs`, and optionally `forced_values`.

In [ ]:
import os

# Define the path to the default.json file
default_json_path = os.path.join(os.getenv('PYTHONPATH'), 'gpuphot', 'instrument_configs', 'default.json')

# Load and display the default configuration as plain text
try:
    with open(default_json_path, 'r') as file:
        default_config_content = file.read()
    print('Default configuration from default.json (with comments):')
    print(default_config_content)
except FileNotFoundError:
    print(f'Error: The file {default_json_path} does not exist.')
except Exception as e:
    print(f'Error reading default.json: {e}')

## Creating a Default Configuration File
The following code demonstrates how to generate a default configuration file.

**Note:** The default configuration is always available internally. However, generating a `default.json` file allows you to customize the baseline settings for all instruments.

In [ ]:
from gpuphot.instrument_config_parser import InstrumentConfigParser
import os

# Define the configuration directory
INSTRUMENT_CONFIG_BASE_PATH = os.getenv('INSTRUMENT_CONFIG_BASE_PATH', '/data/instrument_configs')
print(f'Configuration directory: {INSTRUMENT_CONFIG_BASE_PATH}')

# Create a default configuration file (optional)
config_parser = InstrumentConfigParser(config_dir=INSTRUMENT_CONFIG_BASE_PATH)
config_parser.generate_default_config()
print('Default configuration file generated.')

## Creating a Custom Configuration File
The following code demonstrates how to generate a custom configuration file and open it for editing.

In [ ]:
from gpuphot.instrument_config_parser import InstrumentConfigParser

# Create a custom configuration file
instrument_name = 'my_instrument'
config_parser = InstrumentConfigParser(config_dir=INSTRUMENT_CONFIG_BASE_PATH)
config_parser.generate_config_file(instrument_name)
print(f'Configuration file for {instrument_name} generated.')

# Open the file for editing
file_path = os.path.join(config_parser.config_dir, f'{instrument_name}.json')
with open(file_path, 'r') as file:
    content = file.read()
print(f'Content of {instrument_name}.json:')
print(content)

## Advanced: Forcing Header Values (Overrides)
Sometimes, instrument headers contain incorrect, missing, or unreliable values. You can use the `forced_values` section to strictly override these values.

### Mapping vs. Forcing
*   **`header_keywords` (Mapping):** Use this when the value exists in the header but has a different name (e.g., mapping `EXPTIME` to `EXPOSURE`).
*   **`forced_values` (Overriding):** Use this when you want to ignore the header completely and enforce a specific value (e.g., forcing `GAIN` to `0.33` because the header incorrectly says `1.0`).

Values defined in `forced_values` have **Priority 1**.

In [ ]:
import json

# 1. Load the existing configuration
with open(file_path, 'r') as file:
    config_data = json.load(file)

# 2. Apply a forced value override
# We force 'gain' to 0.33. The system will now use 0.33 even if the FITS header says otherwise.
config_data['forced_values'] = {
    'gain': 0.33
}

# 3. Save the updated configuration
with open(file_path, 'w') as file:
    json.dump(config_data, file, indent=2)

print(f'Updated {instrument_name}.json with forced values.')
print(json.dumps(config_data, indent=2))

## Exploring Configuration Files
The following code explores all the configuration files currently available in your configuration directory.

In [ ]:
import os
import json

# Define the configuration directory
INSTRUMENT_CONFIG_BASE_PATH = os.getenv('INSTRUMENT_CONFIG_BASE_PATH', '/data/instrument_configs')
print(f'Configuration directory: {INSTRUMENT_CONFIG_BASE_PATH}')

# Explore JSON configuration files
for filename in os.listdir(INSTRUMENT_CONFIG_BASE_PATH):
    file_path = os.path.join(INSTRUMENT_CONFIG_BASE_PATH, filename)
    if os.path.isfile(file_path) and filename.endswith('.json'):
        try:
            with open(file_path, 'r') as file:
                data = json.load(file)
                print(f'Content of {filename}:')
                print(json.dumps(data, indent=4))
                print('=' * 50 + '\n')
        except json.JSONDecodeError:
            print(f'Error: {filename} is not a valid JSON file.')
        except Exception as e:
            print(f'Error reading {filename}: {e}')

## Next Steps
Now that you have configured your instrument, proceed to the next notebook to learn how to process images:
- [3. Task Execution](./3_Task_Execution_Notebook.ipynb)